<a href="https://colab.research.google.com/github/Ahmed-kindacool/Week_one_FlyRank/blob/main/w04_baseline_score_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

**Goal:** audit two signals, build one transparent rule-based baseline, write a ranked queue, and review the top 20.

This notebook uses only information available in the starter 90-day snapshot. It does **not** use `trend_direction`, `trend_pct`, or `is_declining_label` as scoring inputs. The label is used only after ranking for the optional Precision@50 receipt.

**Rule idea:** prioritize visible pages that are stale and/or have unusually low CTR for their position. The rule uses simple fixed thresholds — no fitted weights.

In [1]:
# Setup
from pathlib import Path
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/Ahmed-kindacool/Week_one_FlyRank/main/data/raw/content_refresh_anonymized.csv"
LOCAL_DATA = Path("../../data/raw/content_refresh_anonymized.csv")
OUTPUT_PATH = Path("../outputs/baseline_action_score.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

if LOCAL_DATA.exists():
    df = pd.read_csv(LOCAL_DATA)
    data_source = str(LOCAL_DATA)
else:
    df = pd.read_csv(DATA_URL)
    data_source = DATA_URL

print(f"Data source: {data_source}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(df.shape)


Data source: https://raw.githubusercontent.com/Ahmed-kindacool/Week_one_FlyRank/main/data/raw/content_refresh_anonymized.csv
Rows: 30,000
Columns: 44
(30000, 44)


## 1. Signal checks

### Signal 1 — Staleness behind the refresh flag

FlyRank's reference baseline uses a visible-page refresh condition based on `days_since_last_update >= 180` and `impressions_90d >= 500`. We test whether the share of these flag conditions rises across freshness buckets.

**Verdict rule:** CONFIRMED if the observed flag rate increases from younger to older buckets; OPPOSITE if it consistently decreases; MIXED if the pattern is inconsistent; FALSE if the test cannot be computed.

In [2]:
# Signal 1: staleness / refresh-flag audit
d = df.copy()

d["freshness_bucket"] = pd.cut(
    d["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, np.inf],
    labels=["0-30", "31-90", "91-180", "181+"],
    right=True
)

d["stale_visible_flag"] = (
    (d["days_since_last_update"] >= 180) &
    (d["impressions_90d"] >= 500)
).astype(int)

sig1 = (
    d.dropna(subset=["freshness_bucket"])
     .groupby("freshness_bucket", observed=False)
     .agg(n=("stale_visible_flag", "size"),
          flag_rate=("stale_visible_flag", "mean"))
     .reset_index()
)

print(sig1.to_string(index=False))

rates = sig1["flag_rate"].to_numpy()
valid = len(rates) >= 2 and np.all(np.isfinite(rates))
diffs = np.diff(rates) if valid else np.array([])

if not valid:
    verdict1 = "FALSE"
elif np.all(diffs >= 0) and rates[-1] > rates[0]:
    verdict1 = "CONFIRMED"
elif np.all(diffs <= 0) and rates[-1] < rates[0]:
    verdict1 = "OPPOSITE"
else:
    verdict1 = "MIXED"

print(f"Verdict: {verdict1}")


freshness_bucket     n  flag_rate
            0-30 20480   0.000000
           31-90   175   0.000000
          91-180  9171   0.000000
            181+   174   0.097701
Verdict: CONFIRMED


### Signal 2 — CTR versus position behind the CTR-fix logic

The reference baseline treats low CTR as a review signal for pages with impressions and a usable position. We check whether observed CTR generally falls as position gets worse.

`avg_position = 0` means no position data, so those rows are excluded from this signal test.

In [3]:
# Signal 2: CTR versus position
d2 = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 500)].copy()

d2["position_bucket"] = pd.cut(
    d2["avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"],
    right=True
)

sig2 = (
    d2.dropna(subset=["position_bucket"])
      .groupby("position_bucket", observed=False)
      .agg(n=("ctr", "size"),
           mean_ctr=("ctr", "mean"),
           median_ctr=("ctr", "median"))
      .reset_index()
)

print(sig2.to_string(index=False))

ctr_rates = sig2["mean_ctr"].to_numpy()
valid = len(ctr_rates) >= 2 and np.all(np.isfinite(ctr_rates))
diffs = np.diff(ctr_rates) if valid else np.array([])

if not valid:
    verdict2 = "FALSE"
elif np.all(diffs <= 0) and ctr_rates[-1] < ctr_rates[0]:
    verdict2 = "CONFIRMED"
elif np.all(diffs >= 0) and ctr_rates[-1] > ctr_rates[0]:
    verdict2 = "OPPOSITE"
else:
    verdict2 = "MIXED"

print(f"Verdict: {verdict2}")


position_bucket    n  mean_ctr  median_ctr
          top_3  480  0.349500        0.20
         page_1 7084  0.338152        0.24
       striking 4459  0.266629        0.17
       page_3_5 4314  0.142962        0.09
           deep  389  0.043213        0.00
Verdict: CONFIRMED


**Signal decision:** the baseline will use these two signals as fixed, human-readable conditions:

1. **Stale + visible:** `days_since_last_update >= 180` and `impressions_90d >= 500`.
2. **Low CTR + visible:** `avg_position` is between 1 and 20, `impressions_90d >= 500`, and `ctr < 0.5`.

These thresholds are deliberately simple and are not learned from the target label.

## 2. Build the ranked queue

The rule has **one score**, **one reason code per row**, and an **action label**.

- 3 points: stale and visible
- 2 points: low CTR and visible
- 5 points: both conditions
- 0 points: neither condition

Reason codes are mutually exclusive so every row has exactly one reason code.

In [4]:
# Build the transparent baseline score
q = df.copy()

q["stale_visible"] = (
    (q["days_since_last_update"] >= 180) &
    (q["impressions_90d"] >= 500)
).astype(int)

q["low_ctr_visible"] = (
    (q["avg_position"] > 0) &
    (q["avg_position"] <= 20) &
    (q["impressions_90d"] >= 500) &
    (q["ctr"] < 0.5)
).astype(int)

q["score"] = 3 * q["stale_visible"] + 2 * q["low_ctr_visible"]

q["reason_code"] = np.select(
    [
        (q["stale_visible"] == 1) & (q["low_ctr_visible"] == 1),
        q["stale_visible"] == 1,
        q["low_ctr_visible"] == 1,
    ],
    [
        "stale_and_low_ctr",
        "stale_visible_page",
        "low_ctr_visible_page",
    ],
    default="general_review"
)

q["action"] = np.where(q["score"] > 0, "refresh", "monitor")

q = q.sort_values(
    ["score", "impressions_90d", "content_id"],
    ascending=[False, False, True]
).reset_index(drop=True)

q["rank"] = np.arange(1, len(q) + 1)

output_cols = [
    "content_id", "client_id", "rank", "score",
    "reason_code", "action",
    "days_since_last_update", "impressions_90d",
    "avg_position", "ctr", "content_age_days"
]

queue = q[output_cols].copy()
queue.to_csv(OUTPUT_PATH, index=False)

print(f"Wrote: {OUTPUT_PATH}")
print(f"Rows written: {len(queue):,}")
display(queue.head(20))


Wrote: ../outputs/baseline_action_score.csv
Rows written: 30,000


,content_id,client_id,rank,score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr,content_age_days
0,content_cf56e2e2e282,client_7f2253d7e2,1,5,stale_and_low_ctr,refresh,194,61678,19.7,0.15,231
1,content_0a91db491d14,client_7f2253d7e2,2,5,stale_and_low_ctr,refresh,193,13299,10.5,0.49,231
2,content_c2d929d83eaa,client_7f2253d7e2,3,5,stale_and_low_ctr,refresh,193,7558,17.9,0.20,231
3,content_fe16a55cd13d,client_7f2253d7e2,4,5,stale_and_low_ctr,refresh,194,4556,16.4,0.33,231
4,content_928af3e22c80,client_7f2253d7e2,5,5,stale_and_low_ctr,refresh,193,1697,15.8,0.12,231
5,content_e3ff1b093148,client_d029fa3a95,6,5,stale_and_low_ctr,refresh,183,1408,7.8,0.28,232
6,content_7f116ae1f6f5,client_9400f1b21c,7,5,stale_and_low_ctr,refresh,301,954,9.0,0.42,301
7,content_77d4d5930e5e,client_7f2253d7e2,8,5,stale_and_low_ctr,refresh,194,828,18.6,0.24,231
8,content_72496874f806,client_4ec9599fc2,9,5,stale_and_low_ctr,refresh,301,821,5.8,0.24,301
9,content_6226ee6adc91,client_d029fa3a95,10,5,stale_and_low_ctr,refresh,183,545,17.8,0.18,232


In [5]:
# Optional baseline receipt: evaluate only after scoring
if "is_declining_label" in q.columns:
    k = min(50, len(q))
    precision_at_50 = q.head(k)["is_declining_label"].mean()
    base_rate = q["is_declining_label"].mean()
    print(f"Precision@{k}: {precision_at_50:.3f}")
    print(f"Base rate:    {base_rate:.3f}")
    print("The label was NOT used to calculate score, reason_code, action, or rank.")
else:
    print("No label column found; skipping Precision@50 receipt.")


No label column found; skipping Precision@50 receipt.


## 3. Top-20 review

The following review is generated directly from the ranked queue. Each row contains the action, reason code, a confidence note, and what would make the recommendation wrong.

In [6]:
# Generate a one-line skeptical review for the top 20
top20 = q.head(20).copy()

def review_note(row):
    if row["reason_code"] == "stale_and_low_ctr":
        confidence = "moderate-high"
        wrong = "the page is old but does not actually need updating, or its low CTR is explained by intent/brand effects"
    elif row["reason_code"] == "stale_visible_page":
        confidence = "moderate"
        wrong = "age is not causing a problem and the page is already meeting the user's need"
    elif row["reason_code"] == "low_ctr_visible_page":
        confidence = "moderate"
        wrong = "CTR is low for a valid reason such as search intent, SERP features, or brand-query mix"
    else:
        confidence = "low"
        wrong = "the baseline has no strong evidence for intervention"
    return f"Action={row['action']}; reason={row['reason_code']}; confidence={confidence}; wrong if {wrong}."

top20_review = top20[["rank", "content_id", "score", "reason_code", "action"]].copy()
top20_review["review"] = top20.apply(review_note, axis=1)

display(top20_review[["rank", "content_id", "score", "action", "reason_code", "review"]])


,rank,content_id,score,action,reason_code,review
0,1,content_cf56e2e2e282,5,refresh,stale_and_low_ctr,Action=refresh; reason=stale_and_low_ctr; conf...
1,2,content_0a91db491d14,5,refresh,stale_and_low_ctr,Action=refresh; reason=stale_and_low_ctr; conf...
2,3,content_c2d929d83eaa,5,refresh,stale_and_low_ctr,Action=refresh; reason=stale_and_low_ctr; conf...
3,4,content_fe16a55cd13d,5,refresh,stale_and_low_ctr,Action=refresh; reason=stale_and_low_ctr; conf...
4,5,content_928af3e22c80,5,refresh,stale_and_low_ctr,Action=refresh; reason=stale_and_low_ctr; conf...
5,6,content_e3ff1b093148,5,refresh,stale_and_low_ctr,Action=refresh; reason=stale_and_low_ctr; conf...
6,7,content_7f116ae1f6f5,5,refresh,stale_and_low_ctr,Action=refresh; reason=stale_and_low_ctr; conf...
7,8,content_77d4d5930e5e,5,refresh,stale_and_low_ctr,Action=refresh; reason=stale_and_low_ctr; conf...
8,9,content_72496874f806,5,refresh,stale_and_low_ctr,Action=refresh; reason=stale_and_low_ctr; conf...
9,10,content_6226ee6adc91,5,refresh,stale_and_low_ctr,Action=refresh; reason=stale_and_low_ctr; conf...


## 4. Weak picks + leakage check

A useful baseline should have at least one questionable pick. The examples below are deliberately skeptical.

The rule does not use `trend_direction`, `trend_pct`, or `is_declining_label` as inputs. It also does not use any future-window field. The 30-day comparison columns are not used in the score.

In [7]:
# Weak-pick review: show high-scoring rows that rely on a single condition
weak = q[
    (q["score"] > 0) &
    (
        (q["reason_code"] == "stale_visible_page") |
        (q["reason_code"] == "low_ctr_visible_page")
    )
].head(5).copy()

weak["why_weak"] = np.where(
    weak["reason_code"].eq("stale_visible_page"),
    "A page can be old and still be useful; age alone does not establish that a refresh will help.",
    "Low CTR can reflect legitimate search intent or SERP context; the threshold does not prove a CTR problem."
)

display(weak[["rank", "content_id", "score", "reason_code", "action", "why_weak"]])

score_inputs = {
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
}

forbidden_inputs = {"trend_direction", "trend_pct", "is_declining_label"}

print("Score inputs:", sorted(score_inputs))
print("Forbidden target/leakage fields present in score inputs:",
      sorted(score_inputs & forbidden_inputs))
assert not (score_inputs & forbidden_inputs)
print("Leakage check: PASS")


,rank,content_id,score,reason_code,action,why_weak
10,11,content_7368877ea310,3,stale_visible_page,refresh,A page can be old and still be useful; age alo...
11,12,content_1bfaa38ff26c,3,stale_visible_page,refresh,A page can be old and still be useful; age alo...
12,13,content_5feee3994adb,3,stale_visible_page,refresh,A page can be old and still be useful; age alo...
13,14,content_b16bd7307b39,3,stale_visible_page,refresh,A page can be old and still be useful; age alo...
14,15,content_ecb6215e79fd,3,stale_visible_page,refresh,A page can be old and still be useful; age alo...


Score inputs: ['avg_position', 'ctr', 'days_since_last_update', 'impressions_90d']
Forbidden target/leakage fields present in score inputs: []
Leakage check: PASS


## Self-check

- [x] Two signal checks with visible bucket tables and `n`
- [x] At least one signal is linked to a real FlyRank refresh/CTR flag
- [x] Each signal receives a data-driven verdict
- [x] One transparent score with fixed thresholds
- [x] Exactly one reason code per row
- [x] One action label per row
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv`
- [x] Top 20 reviewed with confidence and failure conditions
- [x] Weak picks identified
- [x] No future-window or label-derived inputs used in scoring
- [x] No client names, URLs, private queries, or private client data added

**Interpretation:** this is a decision-support baseline, not a claim about Google's algorithm.